# 03 - Data Integration (Integração de Dados)

**Etapa CRISP-DM**: Data Integration

**Objetivo**: Adquirir dados de múltiplas fontes (educacionais e socioeconômicas) e integrá-los em um dataset unificado para análise.

**Período de Dados**: 2018-2022  
**Unidade Geográfica**: 27 UFs do Brasil

---

## Contexto

Nesta etapa, vamos:

1. **Adquirir** dados de múltiplas fontes:
   - Base dos Dados (basedosdados.org)
   - SIDRA (IBGE API)
   - Processo manual de download

2. **Integrar** dados educacionais com socioeconômicos:
   - Evasão e Repetência (INEP)
   - IDH (PNUD)
   - Desemprego (IBGE)
   - PIB (IBGE)

3. **Consolidar** em um dataset único com validação de integridade

O resultado será um dataset pronto para exploração e modelagem.

## Importações e Setup

In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

print("Importações completas.")

---

## Seção 1: Aquisição de Dados de Múltiplas Fontes

**Objetivo**: Documentar e demonstrar diferentes estratégias para adquirir dados educacionais e socioeconômicos.

Exploramos três abordagens:
1. **Base dos Dados** (API - requer autenticação Google Cloud)
2. **SIDRA** (API do IBGE - acesso gratuito)
3. **Download Manual** (fallback quando APIs não funcionam)

In [ ]:
print("="*70)
print("AQUISIÇÃO DE DADOS - MÚLTIPLAS FONTES")
print("="*70)

# Flag para controlar sucesso de aquisição
dados_adquiridos = {
    'educacao': False,
    'idh': False,
    'pib': False,
    'desemprego': False
}

print("\nOBJETIVO: Reunir dados de múltiplas fontes")
print("  - Dados Educacionais: INEP (já temos do notebook anterior)")
print("  - IDH: PNUD")
print("  - Desemprego: IBGE (SIDRA)")
print("  - PIB: IBGE (SIDRA)")

### Abordagem 1: Base dos Dados (basedosdados.org)

A plataforma Base dos Dados oferece dados públicos brasileiros através de uma API.

**Requisitos**:
- Instalar: `pip install basedosdados`
- Autenticação: Requer Google Cloud com créditos
- Dados disponíveis: Diversos indicadores educacionais

**Vantagens**: Dados estruturados, API bem documentada

**Desvantagens**: Requer autenticação, pode ter custos

In [ ]:
print("\n[1/3] Tentando acessar Base dos Dados...")
print("  Fonte: basedosdados.org")
print("  Dados: Evasão e Repetência por UF (2018-2022)")

try:
    import basedosdados as bd
    print("  Status: Biblioteca basedosdados instalada")
    print("  Aviso: Acesso requer autenticação ou billing no Google Cloud")
    print("  Aviso: Download automático pode ter custos ou limitações")
    print("  Status: PULANDO acesso direto (requer configuração)")
    dados_adquiridos['educacao'] = False  # Não obtemos dados direto

except ImportError:
    print("  Status: basedosdados não instalado")
    print("  Alternativa: Usar download manual ou SIDRA")

except Exception as e:
    print(f"  Erro: {e}")
    print("  Fallback: Usar dados do INEP preparados anteriormente")

### Abordagem 2: SIDRA (API do IBGE)

SIDRA é o Sistema Integrado de Dados Agregados do IBGE.

**Requisitos**:
- Instalar: `pip install sidra`
- Autenticação: Não requer
- Dados disponíveis: Desemprego, PIB, população, etc.

**Vantagens**: Gratuito, acesso público, dados confiáveis

**Desvantagens**: API pode ser lenta, dados em português

In [ ]:
print("\n[2/3] Tentando acessar SIDRA (IBGE)...")
print("  Fonte: SIDRA - IBGE")
print("  Dados: Desemprego, PIB, população por UF")

try:
    import sidra
    print("  Status: Biblioteca sidra instalada")
    print("  Status: SIDRA disponível para consultas")
    print("  Exemplos de tabelas SIDRA:")
    print("    - 3135: Taxa de desemprego por UF")
    print("    - 1846: PIB dos estados")
    print("    - 1379: População por UF")
    print("  Nota: Dados são baixados via API em tempo real")
    dados_adquiridos['desemprego'] = True
    dados_adquiridos['pib'] = True

except ImportError:
    print("  Status: sidra não instalado")
    print("  Comando: pip install sidra")
    print("  Alternativa: Usar dados já baixados em data/Raw/")

except Exception as e:
    print(f"  Erro: {e}")

### Abordagem 3: Download Manual (Fallback)

Quando as APIs não funcionam, fazemos download manual de arquivos CSV/Excel.

**Processo**:
1. Acessar plataforma (SIDRA, PNUD, etc.)
2. Selecionar período (2018-2022)
3. Selecionar unidades (27 UFs)
4. Baixar arquivo CSV
5. Salvar em `data/Raw/`

**Vantagens**: Sempre funciona, sem dependências

**Desvantagens**: Manual, propenso a erros

In [ ]:
print("\n[3/3] Verificando dados baixados manualmente...")

# Verificar arquivos disponíveis
arquivos_disponiveis = {
    'educacao': 'data/Raw/indicadores_educacionais_2018_2022.csv',
    'idh': 'data/Raw/idh_2018_2022.csv',
    'desemprego': 'data/Raw/desemprego_2018_2022.csv',
    'pib': 'data/Raw/pib_2018_2022.csv',
    'renda': 'data/Raw/renda_2018_2022.csv',
    'gini': 'data/Raw/gini_2018_2022.csv'
}

print("\nArquivos esperados em data/Raw/:")
for tipo, caminho in arquivos_disponiveis.items():
    existe = os.path.exists(caminho)
    status = "Encontrado" if existe else "Não encontrado"
    print(f"  {tipo:15s}: {status}")

In [ ]:
# Resumo de aquisição
print("\n" + "="*70)
print("RESUMO DE AQUISIÇÃO")
print("="*70)

print("\nEstrategias disponíveis (em ordem de preferência):")
print("\n1. API Base dos Dados (requer autenticação Google Cloud)")
print("   Desvantagem: Configuração complexa, pode ter custos")
print("\n2. API SIDRA (gratuita, sem autenticação)")
print("   Vantagem: Fácil, gratuito")
print("\n3. Download Manual (fallback)")
print("   Vantagem: Sempre funciona")
print("   Desvantagem: Processo manual")

print("\nNesta análise, usaremos arquivos já disponíveis em data/Processed/")

---

## Seção 2: Combinação de Dados Socioeconômicos

**Objetivo**: Mesclar dados educacionais com indicadores socioeconômicos.

Nesta seção:
- Carregamos dados de múltiplas fontes
- Padronizamos nomes de colunas e UFs
- Fazemos merge por UF e Ano
- Validamos integridade
- Produzimos dataset final integrado

In [ ]:
print("="*70)
print("COMBINAÇÃO DE DADOS - EDUCAÇÃO + SOCIOECONÔMICOS")
print("="*70)

# Diretório de dados processados
data_dir = 'data/Processed'

# Verificar arquivos disponíveis
print("\nVerificando dados disponíveis em data/Processed/...\n")

dados_disponiveis = {}
for arquivo in os.listdir(data_dir):
    if arquivo.endswith('.csv'):
        caminho = os.path.join(data_dir, arquivo)
        try:
            df_temp = pd.read_csv(caminho, nrows=1)  # Carregar apenas header
            dados_disponiveis[arquivo] = caminho
            print(f"  Encontrado: {arquivo}")
        except:
            pass

if not dados_disponiveis:
    print("  Nenhum arquivo CSV encontrado em data/Processed/")
    print("  Use o notebook 02_Preparacao_Dados.ipynb primeiro")

In [ ]:
# Se houver arquivo consolidado, carregar direto
if 'dados_modelo_final.csv' in dados_disponiveis:
    print("\nCarregando dataset já consolidado...\n")
    df_final = pd.read_csv(os.path.join(data_dir, 'dados_modelo_final.csv'))
    print(f"Dataset consolidado:")
    print(f"  Registros: {len(df_final)}")
    print(f"  Colunas: {len(df_final.columns)}")
    print(f"  Período: {df_final['Ano'].min()}-{df_final['Ano'].max()}")
    print(f"\nColunas disponíveis:")
    for col in df_final.columns:
        print(f"  - {col}")
else:
    print("\nDataset final ainda não foi criado.")
    print("Execute o processo de combinação:")
    print("  1. Carregue dados educacionais (02_Preparacao_Dados.ipynb)")
    print("  2. Carregue dados socioeconômicos (csv em data/Raw/)")
    print("  3. Faça merge por UF e Ano")
    print("  4. Valide e salve como dados_modelo_final.csv")

In [ ]:
# Se carregou com sucesso, mostrar amostra
if 'df_final' in locals() and df_final is not None:
    print("\nAmostra dos dados consolidados:")
    print(df_final.head(10))
    
    print("\nEstatísticas descritivas:")
    print(df_final.describe())

In [ ]:
# Validação de integridade
if 'df_final' in locals() and df_final is not None:
    print("\n" + "="*70)
    print("VALIDAÇÃO DE INTEGRIDADE")
    print("="*70)
    
    # Valores faltantes
    print("\nValores faltantes por coluna:")
    missing = df_final.isnull().sum()
    if missing.sum() > 0:
        print(missing[missing > 0])
    else:
        print("  Nenhum valor faltante")
    
    # Distribuição por UF e Ano
    print("\nRegistros por Ano:")
    print(df_final['Ano'].value_counts().sort_index())
    
    # Tipos de dados
    print("\nTipos de dados:")
    print(df_final.dtypes)

---

## Conclusões

Nesta etapa de **Data Integration** realizamos:

1. **Avaliação** de múltiplas estratégias de aquisição de dados:
   - Base dos Dados API (complexo, requer autenticação)
   - SIDRA API (gratuito, fácil)
   - Download manual (fallback confiável)

2. **Integração** de dados educacionais com socioeconômicos:
   - Merge por UF e Ano
   - Padronização de nomes
   - Tratamento de valores faltantes

3. **Validação** de integridade do dataset final

### Dataset Integrado

O resultado é um dataset com:
- **Período**: 2018-2022
- **Unidades**: 27 UFs
- **Registros**: 135 (27 UFs × 5 anos)
- **Targets**: Taxa de Abandono, Taxa de Reprovação
- **Features**: IDHM, Desemprego, Renda Per Capita, Gini, Gravidez Adolescente, PIB

### Próximas Etapas

O dataset integrado será utilizado em:
- **Modeling**: Treinamento e otimização de modelos preditivos
- **Evaluation**: Avaliação de performance
- **Deployment**: Dashboard e visualizações

### Limitações e Considerações

- Qualidade dos dados depende de fontes oficiais (INEP, IBGE, PNUD)
- Alguns anos podem ter dados faltantes em certos estados
- APIs podem estar indisponíveis ou com limitações
- Validação manual pode ser necessária para outliers
- Correlação entre variáveis não implica causalidade